# Sepsis: вероятность повтора и признаки

Examination — состояние, повторы подряд склеиваются только если они в один момент
(`repeats="simultaneous"`). Значения анализов дополнительно лежат в
`log.measurements` и используются как признаки.

1. как вариант подготовки трасс меняет сеть и S(t);
2. вероятность повтора каждого состояния: по модели и по журналу;
3. какие признаки известны на момент входа в состояние;
4. предсказывают ли признаки **повтор** и вообще **выбор выхода** (логистическая регрессия против частот);
5. предсказывают ли они **время до выхода из состояния** (lognormal AFT с цензурой против модели без признаков).

Сравнение по 5 фолдам целых трасс, метрика — средний log L на тестовых интервалах.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

import survpm
from survpm import datasets, features

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160, "display.max_columns", 30, "display.precision", 3)

PATH = Path("../Med_Eremina/Sepsis Cases - Event Log_grouped.csv")
OUT = Path("notebooks/output/features_sepsis")
OUT.mkdir(parents=True, exist_ok=True)
START, TERMINAL = datasets.SEPSIS_START, datasets.SEPSIS_TERMINAL

## 1. Журнал и варианты подготовки трасс

In [2]:
log = datasets.load_sepsis_log(PATH)   # полные трассы, Examination — состояние
log_transparent = datasets.load_sepsis_log(PATH, measurement_activities=datasets.SEPSIS_MEASUREMENTS)

print("статические:", log.static)
print("замеры:", log.dynamic)
print("событий:", len(log.events), "| замеров:", len(log.measurements), "| трасс:", len(log.cases))
log.measurements.groupby("feature")["value"].describe()

статические: ['InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'Age', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG']
замеры: ['CRP', 'LacticAcid', 'Leucocytes']
событий: 13112 | замеров: 7938 | трасс: 1050


,count,mean,std,min,25%,50%,75%,max
feature,,,,,,,,
CRP,3123.0,113.746,86.775,5.0,43.0,95.0,162.0,573.0
LacticAcid,1454.0,1.949,1.479,0.2,1.1,1.6,2.3,14.9
Leucocytes,3361.0,12.949,14.995,0.2,7.7,11.0,15.0,381.3


Три варианта: по умолчанию (`simultaneous`), как раньше (`collapse` — серия Examination склеивается в один заход)
и с прозрачными анализами (Examination не состояние). S(t) — вероятность ещё не дойти до Discharge, сеткой.

In [3]:
variants = {
    "simultaneous (по умолчанию)": survpm.CensoredStateNet(START, TERMINAL).fit(log),
    "collapse (как раньше)": survpm.CensoredStateNet(START, TERMINAL, repeats="collapse").fit(log),
    "анализы прозрачны": survpm.CensoredStateNet(START, TERMINAL).fit(log_transparent),
}
rows = []
for name, m in variants.items():
    iv_ = m.intervals_
    s = m.survival([1, 7, 30], method="grid")
    rows.append({"вариант": name, "заходов": len(iv_), "переходов": len(m.edges_),
                 "петель s→s": sum(e["src"] == e["dst"] for e in m.edges_),
                 "S(1)": s[0], "S(7)": s[1], "S(30)": s[2]})
pd.DataFrame(rows).set_index("вариант")

,заходов,переходов,петель s→s,S(1),S(7),S(30)
вариант,,,,,,
simultaneous (по умолчанию),7543,15,3,0.925,0.427,0.013
collapse (как раньше),5090,12,0,0.942,0.425,0.010
анализы прозрачны,3859,8,2,0.968,0.445,0.010


In [4]:
model = variants["simultaneous (по умолчанию)"]
iv = model.intervals_
model.edges_table()[["transition", "p", "distribution", "n_completed", "median_days"]]

,transition,p,distribution,n_completed,median_days
0,Admission→Admission,0.156,exponential(mean=0.604046),201,0.419
1,Admission→Discharge,0.108,"weibull(shape=1.36145, scale=3.51124)",139,2.683
2,Admission→Examination,0.713,"weibull(shape=1.10792, scale=1.26416)",916,0.908
3,Admission→Treatment,0.023,"gamma(shape=0.476013, scale=1.08694)",29,0.225
4,Examination→Admission,0.140,"lognormal(median=0.167604, sigma=1.10164)",499,0.168
5,Examination→Discharge,0.179,"gamma(shape=1.08946, scale=1.91736)",637,1.496
6,Examination→Examination,0.484,"gamma(shape=1.49463, scale=1.05211)",1724,1.239
7,Examination→Treatment,0.198,"gamma(shape=0.851631, scale=0.0887529)",705,0.049
8,Registration→Admission,0.007,exponential(mean=0.11956),7,0.083
9,Registration→Examination,0.748,"lognormal(median=0.0162862, sigma=0.604894)",758,0.016


## 2. Вероятность повтора

По модели (`model.repeat_table()`, из вероятностей выбора, оценённых с учётом цензуры):
`p_immediate` — следующим будет то же состояние, `p_return` — состояние ещё встретится до Discharge,
`expected_visits` — среднее число заходов. По журналу — те же доли среди заходов в завершённых трассах
(у незавершённых будущее неизвестно, поэтому только завершённые; это смещает оценку вниз).

In [5]:
def empirical_repeats(iv, cases_done):
    rows = []
    for case_id, g in iv.loc[iv["case_id"].isin(cases_done)].groupby("case_id", sort=False):
        seq = g["src"].tolist() + [g["dst"].iloc[-1]]
        for k, s in enumerate(seq[:-1]):
            rows.append(dict(state=s, immediate=seq[k + 1] == s, later=s in seq[k + 1:]))
    table = pd.DataFrame(rows).groupby("state").agg(n=("immediate", "size"), p_immediate_log=("immediate", "mean"),
                                                    p_return_log=("later", "mean"))
    return table


done_cases = iv.groupby("case_id")["observed"].all()
done_cases = done_cases.index[done_cases]
repeats = model.repeat_table().set_index("state").join(empirical_repeats(iv, done_cases))
repeats.to_csv(OUT / "repeats.csv")
repeats

,p_immediate,p_return,expected_visits,n,p_immediate_log,p_return_log
state,,,,,,
Admission,0.156,0.605,2.531,1270,0.158,0.385
Examination,0.484,0.784,4.638,3418,0.501,0.772
Registration,0.000,0.000,1.000,782,0.000,0.000
Treatment,0.371,0.665,2.987,1250,0.346,0.456


Для сравнения: при `collapse` петли Examination → Examination нет, и вероятность повтора Examination
берётся только из возвратов через другие состояния.

In [6]:
variants["collapse (как раньше)"].repeat_table().set_index("state")

,p_immediate,p_return,expected_visits
state,,,
Admission,0.0,0.529,2.124
Examination,0.0,0.580,2.380
Registration,0.0,0.000,1.000
Treatment,0.0,0.465,1.868


## 3. Признаки на момент входа в состояние

Значение анализа, взятого в момент входа в Examination, считается известным на входе: оно описывает
этот же заход.

In [7]:
X = features.at_entry(iv, log, window="1D")
data = iv.join(X)
coverage = data.groupby("src")[["CRP", "Leucocytes", "LacticAcid"]].agg(lambda s: s.notna().mean())
coverage.insert(0, "n", data.groupby("src").size())
coverage

,n,CRP,Leucocytes,LacticAcid
src,,,,
Admission,1298,0.951,0.994,0.902
Examination,3684,0.964,0.999,0.892
Registration,1050,0.021,0.027,0.003
Treatment,1511,0.756,0.782,0.732


Модельные признаки: возраст, флаги SIRS и диагностики (статические), последние CRP, Leucocytes,
LacticAcid в log-шкале с флагом «ещё не измеряли», число анализов за сутки. Пропуски — медиана
плюс индикатор; всё стандартизуется по обучающему фолду.

In [8]:
DYNAMIC = ["CRP", "Leucocytes", "LacticAcid"]
STATIC = [c for c in log.static if pd.api.types.is_numeric_dtype(log.cases[c])]


def design(frame):
    cols = {c: frame[c].astype(float) for c in STATIC}
    for f in DYNAMIC:
        cols[f"log_{f}"] = np.log1p(frame[f].astype(float))
        cols[f"{f}_missing"] = frame[f].isna().astype(float)
        cols[f"{f}_n_1D"] = frame[f"{f}_n_1D"].astype(float)
    return pd.DataFrame(cols, index=frame.index)


class Prep:
    """Медиана для пропусков и стандартизация, параметры — по обучающей части. Признаки, у которых
    от самого частого значения отличаются меньше 10 строк, выбрасываются: на тесте они дают
    огромные стандартизованные значения."""
    def fit(self, Z):
        self.median = Z.median()
        filled = Z.fillna(self.median)
        minority = filled.apply(lambda s: (s != s.mode().iloc[0]).sum())
        self.columns = minority.index[minority >= 10]
        filled = filled[self.columns]
        self.mean, self.std = filled.mean(), filled.std()
        return self

    def __call__(self, Z):
        Z = Z[self.columns].fillna(self.median[self.columns])
        return ((Z - self.mean) / self.std).clip(-5, 5).to_numpy()


Z = design(data)
print(Z.shape[1], "признаков:", list(Z.columns))

32 признаков: ['InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'Age', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG', 'log_CRP', 'CRP_missing', 'CRP_n_1D', 'log_Leucocytes', 'Leucocytes_missing', 'Leucocytes_n_1D', 'log_LacticAcid', 'LacticAcid_missing', 'LacticAcid_n_1D']


## 4. Повтор и выбор выхода

Для каждого состояния: логистическая регрессия против частот (это и есть нынешнее `p`), только
завершённые интервалы — у цензурированных выход неизвестен. Сначала цель «следующее состояние — то же
самое» (повтор), потом весь выбор выхода `dst ~ x`.

In [9]:
folds = GroupKFold(n_splits=5)


def choice_cv(Z, repeat=False):
    """``repeat=True`` — цель «повтор / другое», иначе следующее состояние."""
    rows, coefs = [], {}
    done = data.loc[data["observed"]]
    for src, group in done.groupby("src"):
        target = np.where(group["dst"] == src, "повтор", "другое") if repeat else group["dst"].to_numpy()
        if len(set(target)) < 2 or (repeat and (target == "повтор").sum() < 10):
            continue
        y, Zs, cases = target, Z.loc[group.index], group["case_id"].to_numpy()
        classes = np.unique(y)
        ll_base, ll_x, n = 0., 0., 0
        for train, test in folds.split(Zs, y, cases):
            prior = pd.Series(y[train]).value_counts(normalize=True).reindex(classes).fillna(1e-6).to_numpy()
            prep = Prep().fit(Zs.iloc[train])
            clf = LogisticRegression(C=1.0, max_iter=2000).fit(prep(Zs.iloc[train]), y[train])
            proba = pd.DataFrame(clf.predict_proba(prep(Zs.iloc[test])), columns=clf.classes_)
            proba = proba.reindex(columns=classes).fillna(1e-6).to_numpy()
            idx = np.searchsorted(classes, y[test])
            ll_base += np.log(prior[idx]).sum()
            ll_x += np.log(np.clip(proba[np.arange(len(test)), idx], 1e-12, 1)).sum()
            n += len(test)
        rows.append(dict(src=src, exits=", ".join(f"{c} ({(y == c).mean():.0%})" for c in classes), n=n,
                         logL_const=ll_base / n, logL_x=ll_x / n, gain=(ll_x - ll_base) / n))
        prep = Prep().fit(Zs)
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(prep(Zs), y)
        coefs[src] = pd.DataFrame(clf.coef_, columns=prep.columns,
                                  index=clf.classes_ if len(classes) > 2 else [f"{clf.classes_[1]} vs {clf.classes_[0]}"])
    return pd.DataFrame(rows), coefs


repeat_cv, repeat_coefs = choice_cv(Z, repeat=True)
repeat_cv.to_csv(OUT / "repeat_cv.csv", index=False)
repeat_cv

,src,exits,n,logL_const,logL_x,gain
0,Admission,"другое (84%), повтор (16%)",1285,-0.434,-0.451,-0.017
1,Examination,"другое (52%), повтор (48%)",3565,-0.693,-0.672,0.021
2,Treatment,"другое (63%), повтор (37%)",1411,-0.662,-0.653,0.009


Признаки, сильнее всего связанные с повтором (стандартизованные коэффициенты, «+» — повтор вероятнее):

In [10]:
for src, table in repeat_coefs.items():
    sign = 1 if table.index[0] == "повтор vs другое" else -1
    row = sign * table.iloc[0]
    print(src)
    display(row.reindex(row.abs().sort_values(ascending=False).index[:8]).round(2).to_frame("coef").T)

Admission


,InfectionSuspected,Hypotensie,DiagnosticArtAstrup,DiagnosticUrinaryCulture,DiagnosticBlood,SIRSCriteria2OrMore,SIRSCritTemperature,log_LacticAcid
coef,0.2,-0.17,0.14,-0.13,0.12,-0.11,-0.09,0.08


Examination


,Hypotensie,log_CRP,log_LacticAcid,DiagnosticECG,DiagnosticXthorax,InfectionSuspected,DisfuncOrg,DiagnosticUrinarySediment
coef,0.42,0.38,-0.22,-0.21,0.21,0.19,-0.16,-0.15


Treatment


,Infusion,DiagnosticOther,Leucocytes_missing,DiagnosticIC,DiagnosticBlood,SIRSCritTemperature,DiagnosticArtAstrup,Hypotensie
coef,0.45,-0.31,-0.18,0.13,-0.11,-0.11,0.08,-0.08


Весь выбор выхода:

In [11]:
choice, coefs = choice_cv(Z)
choice.to_csv(OUT / "choice_cv.csv", index=False)
choice

,src,exits,n,logL_const,logL_x,gain
0,Admission,"Admission (16%), Discharge (11%), Examination ...",1285,-0.860,-0.878,-0.018
1,Examination,"Admission (14%), Discharge (18%), Examination ...",3565,-1.255,-1.192,0.063
2,Registration,"Admission (1%), Examination (75%), Treatment (...",1014,-0.599,-0.565,0.033
3,Treatment,"Admission (42%), Discharge (0%), Examination (...",1411,-1.081,-0.808,0.273


Самые сильные признаки (стандартизованные коэффициенты на всём журнале):

In [12]:
for src, table in coefs.items():
    top = table.abs().max().sort_values(ascending=False).index[:6]
    print(src)
    display(table[top].round(2))

Admission


,SIRSCritTemperature,Infusion,log_CRP,DiagnosticIC,Oligurie,DiagnosticUrinaryCulture
Admission,-0.28,-0.19,0.06,0.21,0.24,-0.10
Discharge,-0.21,-0.32,-0.44,-0.10,-0.08,-0.28
Examination,-0.20,-0.12,0.05,0.26,0.20,0.07
Treatment,0.70,0.63,0.33,-0.37,-0.36,0.31


Examination


,SIRSCriteria2OrMore,DiagnosticOther,log_CRP,Hypotensie,log_Leucocytes,log_LacticAcid
Admission,-0.25,-0.52,0.11,0.08,0.04,-0.05
Discharge,-0.17,0.22,-0.34,-0.18,-0.23,-0.06
Examination,-0.12,0.03,0.27,0.31,-0.07,-0.15
Treatment,0.54,0.27,-0.05,-0.21,0.26,0.26


Registration


,Infusion,DiagnosticXthorax,DiagnosticLacticAcid,CRP_missing,SIRSCritHeartRate,DiagnosticECG
Admission,-0.55,-0.53,-0.50,-0.49,-0.43,-0.42
Examination,-0.00,0.21,0.08,0.36,0.13,0.08
Treatment,0.55,0.32,0.43,0.13,0.30,0.34


Treatment


,Leucocytes_missing,DiagnosticUrinarySediment,log_LacticAcid,DiagnosticArtAstrup,LacticAcid_missing,Hypotensie
Admission,-1.12,0.24,0.11,-0.19,-0.27,-0.07
Discharge,-0.17,-0.71,-0.64,0.63,-0.18,0.57
Examination,0.92,0.25,0.34,-0.31,0.60,-0.29
Treatment,0.38,0.22,0.19,-0.14,-0.15,-0.22


Флаг `Infusion` записан в регистрации, но по смыслу почти совпадает с последующим шагом Treatment
(IV Liquid). Без него выигрыш на Registration такой:

In [13]:
choice_no_infusion, _ = choice_cv(Z.drop(columns=["Infusion"]))
choice_no_infusion[["src", "n", "gain"]]

,src,n,gain
0,Admission,1285,-0.018
1,Examination,3565,0.063
2,Registration,1014,0.029
3,Treatment,1411,0.266


## 5. Время до выхода из состояния

Для каждого состояния — время пребывания до любого выхода, цензурированные пребывания учитываются как
`S(t)`. Модель — lognormal AFT: `log T = a + b·x + σ·ε`. Сравнение с той же моделью без признаков
(`b = 0`) по log L на тестовых фолдах. Небольшая L2-регуляризация, чтобы десятки флагов не переобучались.

In [14]:
def fit_aft(t, observed, Zm, l2=1.0):
    k = Zm.shape[1]
    logt = np.log(t)

    def nll(theta):
        a, b, log_s = theta[0], theta[1:k + 1], theta[-1]
        s = np.exp(log_s)
        z = (logt - a - Zm @ b) / s
        ll = np.where(observed, stats.norm.logpdf(z) - log_s - logt, stats.norm.logsf(z))
        return -ll.sum() + l2 * (b @ b) / 2

    theta0 = np.r_[logt.mean(), np.zeros(k), np.log(logt.std() + 1e-3)]
    return optimize.minimize(nll, theta0, method="L-BFGS-B").x


def aft_loglik(theta, t, observed, Zm):
    k = Zm.shape[1]
    a, b, s = theta[0], theta[1:k + 1], np.exp(theta[-1])
    z = (np.log(t) - a - Zm @ b) / s
    return np.where(observed, stats.norm.logpdf(z) - np.log(s) - np.log(t), stats.norm.logsf(z)).sum()


rows, time_coefs = [], {}
for src, group in data.groupby("src"):
    if group["observed"].sum() < 30:
        continue
    t, obs = group["duration"].to_numpy(), group["observed"].to_numpy()
    Zs, cases = Z.loc[group.index], group["case_id"].to_numpy()
    ll0 = ll1 = 0.
    for train, test in folds.split(Zs, obs, cases):
        prep = Prep().fit(Zs.iloc[train])
        empty = np.zeros((len(train), 0))
        th0 = fit_aft(t[train], obs[train], empty)
        th1 = fit_aft(t[train], obs[train], prep(Zs.iloc[train]))
        ll0 += aft_loglik(th0, t[test], obs[test], np.zeros((len(test), 0)))
        ll1 += aft_loglik(th1, t[test], obs[test], prep(Zs.iloc[test]))
    rows.append(dict(src=src, n=len(group), censored=int((~obs).sum()), median_days=np.median(t[obs]),
                     logL_const=ll0 / len(group), logL_x=ll1 / len(group), gain=(ll1 - ll0) / len(group)))
    prep = Prep().fit(Zs)
    theta = fit_aft(t, obs, prep(Zs))
    time_coefs[src] = pd.Series(theta[1:-1], index=prep.columns)
timing = pd.DataFrame(rows)
timing.to_csv(OUT / "time_cv.csv", index=False)
timing

,src,n,censored,median_days,logL_const,logL_x,gain
0,Admission,1298,13,0.852,-1.409,-1.393,0.015
1,Examination,3684,119,1.000,-1.145,-1.119,0.026
2,Registration,1050,36,0.015,2.995,3.021,0.026
3,Treatment,1511,100,0.002,2.905,2.897,-0.008


Коэффициенты `b` (стандартизованные): `exp(b)` — во сколько раз растягивается время при росте признака на одно std.

In [15]:
tc = pd.DataFrame(time_coefs).T
top = tc.abs().max().sort_values(ascending=False).index[:8]
np.exp(tc[top]).round(2)

,Leucocytes_missing,log_LacticAcid,log_Leucocytes,InfectionSuspected,CRP_missing,DiagnosticLiquor,Age,Hypotensie
Admission,NaN,0.75,1.00,0.88,1.10,NaN,1.07,0.93
Examination,NaN,0.76,0.77,1.29,0.78,0.89,1.19,1.17
Registration,0.97,NaN,1.04,1.04,0.88,NaN,0.94,0.99
Treatment,0.71,0.96,0.87,1.03,1.23,0.82,0.89,1.01


## 6. Что получилось

In [16]:
print("Повтор, прирост log L на интервал:")
print(repeat_cv[["src", "n", "gain"]].to_string(index=False))
print()
print("Выбор выхода, прирост log L на интервал:")
print(choice[["src", "n", "gain"]].to_string(index=False))
print()
print("Время до выхода, прирост log L на пребывание:")
print(timing[["src", "n", "gain"]].to_string(index=False))

Повтор, прирост log L на интервал:
        src    n   gain
  Admission 1285 -0.017
Examination 3565  0.021
  Treatment 1411  0.009

Выбор выхода, прирост log L на интервал:
         src    n   gain
   Admission 1285 -0.018
 Examination 3565  0.063
Registration 1014  0.033
   Treatment 1411  0.273

Время до выхода, прирост log L на пребывание:
         src    n   gain
   Admission 1298  0.015
 Examination 3684  0.026
Registration 1050  0.026
   Treatment 1511 -0.008


**Выводы (Sepsis, полные трассы, 5 фолдов по трассам; прирост — log L на интервал):**

- Без склейки повторов заходов 7543 вместо 5090, появились петли Admission, Examination, Treatment.
  S(t) до Discharge почти не меняется (S(7) = 0.427 против 0.425).
- **Повтор Examination** модель описывает хорошо: следующим снова Examination — 0.48 (в журнале 0.50),
  Examination ещё встретится — 0.78 (0.77), в среднем 4.6 захода.
- **Admission и Treatment:** немедленный повтор совпадает с журналом (0.16 и 0.37), а вероятность
  вернуться позже модель завышает (0.61 против 0.39 и 0.67 против 0.46). Причина — модель без памяти:
  после ухода из состояния путь не зависит от того, что уже было. Здесь и нужна история трассы.
- **Признаки на входе → немедленный повтор:** почти ничего (Examination +0.02, Treatment +0.01,
  Admission — нет). В Examination повтор вероятнее при высоком CRP и гипотензии.
- **Признаки → весь выбор выхода:** заметно в Treatment (+0.27, главный признак — был ли уже анализ),
  в Examination +0.06, в Registration +0.03.
- **Время:** выигрыш маленький (до +0.03).

Что дальше: для вероятности повтора важнее признаки истории (сколько раз уже был в состоянии,
сколько дней от начала), а не лабораторные значения. Их можно добавить как замеры через
`log.add_measurements` и проверить тем же способом.